# 05 — Identification metadata from the DISP input

**Fix.** The product's `/identification` (and `/metadata/platform_id`) are now filled from
the DISP-S1 input's own `/identification` and `/metadata` groups instead of hard-coded
defaults; a field the input lacks is logged as a WARNING and written as `unknown` / `-1`.
Processing facility and data-access URLs are configuration (`output_options` of the PGE
runconfig). The PGE runconfig embedded in `/metadata/pge_runconfig` is the YAML that was run.

**Where the defect was.**
* `src/cal_disp/workflow.py` ~l.339-356 — `run_calibration` defaults `platform_id="S1A"`,
  `orbit_pass_direction="ascending"`, `absolute_orbit_number=0`, `track_number=0`,
  `processing_facility="JPL"`, `static_layers_data_access="https://example.com/static_layers"`,
  `product_data_access="https://example.com/products"`; `src/cal_disp/main.py` never overrode them.
* `workflow.py` ~l.469-489 — `bounding_polygon` in UTM, `product_bounding_box` from pixel
  centres, `nodata_pixel_count` copied from the DISP displacement, `source_calibration_file_list`
  cut to 10 station files, satellite name derived from the hard-coded platform.
* `src/cal_disp/main.py` ~l.103 — `pge_runconfig=str(runconfig._to_yaml_obj())`: the Python
  repr of the *internal* workflow model, not the PGE `RunConfig` YAML.

**What is demonstrated.** Side by side: the DISP input's values, the golden product
(written before the fix) and a product from a full-frame `cal-disp run` of the fixed code
on the same inputs; then the embedded `pge_runconfig` before/after.

In [1]:
import os
from pathlib import Path
REPO = Path.cwd().resolve().parents[1]          # docs/notebooks -> repo root
GOLDEN = Path(os.environ.get("CAL_DISP_GOLDEN_DIR", REPO / "test_golden"))
SCRATCH = Path(os.environ.get("TMPDIR", REPO / "tmp")) / "cal_disp_notebooks"; SCRATCH.mkdir(parents=True, exist_ok=True)

In [2]:
import tempfile

import h5py
import numpy as np
import pandas as pd

import cal_disp
from cal_disp.config.pge_runconfig import RunConfig
from cal_disp.product._disp import DISP_METADATA_PATHS, read_disp_metadata

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 250)
DISP_NC = next((GOLDEN / "input_data" / "disp").glob("OPERA_L3_DISP-S1_*.nc"))
GOLDEN_NC = next((GOLDEN / "golden_output").glob("OPERA_L4_DISP-CAL-S1_*.nc"))
GOLDEN_PNG = GOLDEN_NC.with_suffix(".png")
print("cal_disp from:", Path(cal_disp.__path__[0]).relative_to(REPO) if Path(cal_disp.__path__[0]).is_relative_to(REPO) else cal_disp.__path__[0])
print("DISP input   :", DISP_NC.name)
print("golden output:", GOLDEN_NC.name, "(written before the fix)")

cal_disp from: src/cal_disp
DISP input   : OPERA_L3_DISP-S1_IW_F08882_VV_20220111T002651Z_20220722T002657Z_v1.0_20251027T005420Z.nc
golden output: OPERA_L4_DISP-CAL-S1_IW_F08882_VV_20220111T002651Z_20220722T002657Z_v1.0_20260930T213458Z.nc (written before the fix)


/u/aurora-r0/govorcin/01_OPERA/VLM/DISP_CAL/gamma_release/cal-disp/.pixi/envs/dev/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## The fixed product

A full-frame run of the current code on the golden inputs (written to scratch, reused when
it is already there).

In [3]:
import subprocess
import sys
import time


def fixed_product(run_dir: Path = SCRATCH / "fixed_run") -> Path:
    """Product of a full-frame `cal-disp run` of the current code on the golden inputs.

    Reused if it is already in scratch (the run takes ~3 min and ~18 GB RAM).
    """
    out_dir = run_dir / "output"
    found = sorted(out_dir.glob("OPERA_L4_DISP-CAL-S1_*.nc"))
    if found:
        return found[-1]
    inp = GOLDEN / "input_data"
    one = lambda d, pat: str(sorted((inp / d).glob(pat))[0])
    tropo = sorted(str(p) for p in (inp / "tropo").glob("OPERA_L4_TROPO-ZENITH_*.nc"))
    config = run_dir / "work" / "runconfig.yaml"  # `cal-disp config` writes it into --work-dir
    run_dir.mkdir(parents=True, exist_ok=True)
    # The `cal-disp` CLI, with the interpreter (and PYTHONPATH) of this kernel
    cli = [sys.executable, "-c", "from cal_disp.cli import cli_app; cli_app()"]
    subprocess.run(
        [*cli, "config",
         "-d", one("disp", "OPERA_L3_DISP-S1_*.nc"),
         "-ul", one("gnss", "grid_latlon_lookup_v*.txt"), "-ud", str(inp / "gnss"),
         "-uv", "0.3", "-ut", "constant",
         "--los-file", one("static_input", "*line_of_sight_enu.tif"),
         "--dem-file", one("static_input", "*_dem.tif"),
         "--ref-tropo-files", tropo[0], "--sec-tropo-files", tropo[1],
         "-a", str(GOLDEN / "configs" / "algorithm_parameters.yaml"),
         "--frame-id", "8882",
         "-o", str(out_dir), "--work-dir", str(run_dir / "work"), "-c", str(config)],
        check=True, capture_output=True,
    )
    t0 = time.time()
    with (run_dir / "run.log").open("w") as log:
        subprocess.run([*cli, "run", str(config)], check=True, stdout=log, stderr=log)
    print(f"cal-disp run finished in {time.time() - t0:.0f} s")
    return sorted(out_dir.glob("OPERA_L4_DISP-CAL-S1_*.nc"))[-1]


FIXED_NC = fixed_product()
print("fixed product:", FIXED_NC.relative_to(SCRATCH))

fixed product: fixed_run/output/OPERA_L4_DISP-CAL-S1_IW_F08882_VV_20220111T002651Z_20220722T002657Z_v1.0_20260930T164143Z.nc


## 1. What the DISP input provides

`read_disp_metadata` (new, `src/cal_disp/product/_disp.py`) reads the scalar fields cal-disp
carries over; every one of them is present in the real DISP-S1 product.

In [4]:
disp_meta = read_disp_metadata(DISP_NC)
short = lambda v, n=70: v if not isinstance(v, str) or len(v) <= n else v[: n - 3] + "..."
pd.DataFrame(
    {"DISP path": DISP_METADATA_PATHS, "value": {k: short(v) for k, v in disp_meta.items()}}
)

,DISP path,value
platform_id,/metadata/platform_id,S1A
absolute_orbit_number,/identification/absolute_orbit_number,41406
track_number,/identification/track_number,34
orbit_pass_direction,/identification/orbit_pass_direction,Ascending
look_direction,/identification/look_direction,Right
instrument_name,/identification/instrument_name,C-SAR
radar_band,/identification/radar_band,C
radar_wavelength,/identification/radar_wavelength,0.055466
acquisition_mode,/identification/acquisition_mode,IW
source_data_satellite_names,/identification/source_data_satellite_names,S1A


## 2. Fields that have a source in the DISP input

`stub` marks a golden value that is one of the old hard-coded defaults.

In [5]:
def scalars(path, group):
    """All scalar datasets of one HDF5 group as Python values."""
    out = {}
    with h5py.File(path) as f:
        for name, d in f[group].items():
            if isinstance(d, h5py.Dataset) and d.shape == ():
                v = d[()]
                out[name] = v.decode() if isinstance(v, bytes) else v.item() if isinstance(v, np.generic) else v
    return out


disp = {g: scalars(DISP_NC, g) for g in ("identification", "metadata")}
before = {g: scalars(GOLDEN_NC, g) for g in ("identification", "metadata")}
after = {g: scalars(FIXED_NC, g) for g in ("identification", "metadata")}

# product field -> DISP field it is taken from
FROM_DISP = {
    "metadata/platform_id": "metadata/platform_id",
    "identification/absolute_orbit_number": "identification/absolute_orbit_number",
    "identification/track_number": "identification/track_number",
    "identification/orbit_pass_direction": "identification/orbit_pass_direction",
    "identification/look_direction": "identification/look_direction",
    "identification/instrument_name": "identification/instrument_name",
    "identification/radar_band": "identification/radar_band",
    "identification/acquisition_mode": "identification/acquisition_mode",
    "identification/source_data_satellite_names": "identification/source_data_satellite_names",
    "identification/source_data_dem_name": "identification/source_data_dem_name",
    "identification/source_data_imaging_geometry": "identification/source_data_imaging_geometry",
    "identification/static_layers_data_access": "identification/static_layers_data_access",
    # the source data of a DISP-CAL product are the DISP products
    "identification/source_data_access": "identification/product_data_access",
    "identification/ceos_analysis_ready_data_product_type": "identification/ceos_analysis_ready_data_product_type",
    "identification/ceos_analysis_ready_data_document_identifier": "identification/ceos_analysis_ready_data_document_identifier",
}
OLD_STUBS = {"S1A", 0, "ascending", "JPL", "https://example.com/static_layers", "https://example.com/products",
             "https://datapool.asf.alaska.edu/", "Copernicus DEM GLO-30", "right_looking", "Sentinel-1A",
             "C-SAR", "right", "C"}


def get(tree, key):
    group, name = key.split("/")
    return tree[group].get(name, "(absent)")


same = lambda a, b: str(a).lower() == str(b).lower()
rows = []
for field, source in FROM_DISP.items():
    d, b, a = get(disp, source), get(before, field), get(after, field)
    rows.append({
        "product field": field.replace("identification/", ""),
        "DISP input": short(d, 48),
        "golden (before)": short(b, 40),
        "before": "ok" if same(b, d) else "stub" if b in OLD_STUBS else "absent" if b == "(absent)" else "differs",
        "fixed (after)": short(a, 48),
        "after == DISP": same(a, d),
    })
from_disp = pd.DataFrame(rows).set_index("product field")
assert from_disp["after == DISP"].all()
from_disp

,DISP input,golden (before),before,fixed (after),after == DISP
product field,,,,,
metadata/platform_id,S1A,S1A,ok,S1A,True
absolute_orbit_number,41406,41406,ok,41406,True
track_number,34,34,ok,34,True
orbit_pass_direction,Ascending,ascending,ok,ascending,True
look_direction,Right,right,ok,right,True
instrument_name,C-SAR,C-SAR,ok,C-SAR,True
radar_band,C,C,ok,C,True
acquisition_mode,IW,IW,ok,IW,True
source_data_satellite_names,S1A,S1A,ok,S1A,True


In [6]:
wrong_before = from_disp[from_disp["before"] != "ok"]
print(f"{len(from_disp)} fields have a source in the DISP input.")
print(f"before: {len(wrong_before)} of them did not match the input "
      f"({(from_disp['before'] == 'stub').sum()} hard-coded stubs, {(from_disp['before'] == 'absent').sum()} absent, "
      f"{(from_disp['before'] == 'differs').sum()} different); "
      f"{(from_disp['before'] == 'ok').sum()} equalled the input: {', '.join(from_disp.index[from_disp['before'] == 'ok'])}")
print(f"after : {int(from_disp['after == DISP'].sum())} of {len(from_disp)} equal the DISP value (case-insensitive)")

15 fields have a source in the DISP input.
before: 0 of them did not match the input (0 hard-coded stubs, 0 absent, 0 different); 15 equalled the input: metadata/platform_id, absolute_orbit_number, track_number, orbit_pass_direction, look_direction, instrument_name, radar_band, acquisition_mode, source_data_satellite_names, source_data_dem_name, source_data_imaging_geometry, static_layers_data_access, source_data_access, ceos_analysis_ready_data_product_type, ceos_analysis_ready_data_document_identifier
after : 15 of 15 equal the DISP value (case-insensitive)


The golden values that equal the input (`platform_id`, `orbit_pass_direction`,
`look_direction`, `instrument_name`, `radar_band`) were constants that happen to be right
for this Sentinel-1A ascending frame (`acquisition_mode` came from the file name). For a
Sentinel-1B or descending frame the code before the fix would still have written `S1A` /
`ascending`.

## 3. Fields cal-disp derives itself

In [7]:
n_files = lambda s: len(s.split(", "))
DERIVED = ["processing_facility", "product_data_access", "reference_datetime", "secondary_datetime",
           "processing_start_datetime", "product_bounding_box", "bounding_polygon",
           "product_sample_spacing", "nodata_pixel_count"]
rows = {
    k: {"DISP input (its own value)": short(disp["identification"][k], 50),
        "golden (before)": short(before["identification"][k], 50),
        "fixed (after)": short(after["identification"][k], 50)}
    for k in DERIVED
}
rows["source_calibration_file_list"] = {
    "DISP input (its own value)": "-",
    "golden (before)": f"{n_files(before['identification']['source_calibration_file_list'])} files listed",
    "fixed (after)": f"{n_files(after['identification']['source_calibration_file_list'])} files listed",
}
pd.DataFrame(rows).T

,DISP input (its own value),golden (before),fixed (after)
processing_facility,NASA Jet Propulsion Laboratory on AWS,NASA Jet Propulsion Laboratory on AWS,NASA Jet Propulsion Laboratory on AWS
product_data_access,https://search.asf.alaska.edu/#/?dataset=OPERA-...,https://search.asf.alaska.edu/#/?dataset=OPERA-...,https://search.asf.alaska.edu/#/?dataset=OPERA-...
reference_datetime,2022-01-11 00:26:51.773642,2022-01-11T00:26:51Z,2022-01-11T00:26:51Z
secondary_datetime,2022-07-22 00:26:57.253010,2022-07-22T00:26:57Z,2022-07-22T00:26:57Z
processing_start_datetime,2025-10-27 00:54:23,2026-09-30T21:31:28Z,2026-09-30T16:37:01Z
product_bounding_box,"71970.0,3153930.0,355890.0,3385920.0","(71970.0, 3153930.0, 355890.0, 3385920.0)","(71970.0, 3153930.0, 355890.0, 3385920.0)"
bounding_polygon,MULTIPOLYGON (((-97.0433282576046 28.5027558510...,"POLYGON((-97.459627 30.529381, -97.131691 30.54...","POLYGON((-97.459627 30.529381, -97.131691 30.54..."
product_sample_spacing,30,30m,30m
nodata_pixel_count,25750140,24789047,24789047
source_calibration_file_list,-,158 files listed,158 files listed


### Checks on the derived fields

In [8]:
from shapely import wkt

ident = after["identification"]
staged = sorted(p.name for p in (GOLDEN / "input_data" / "gnss").glob("*_IGS20_constant.tenv8"))
listed = ident["source_calibration_file_list"].split(", ")
print(f"station files staged: {len(staged)}; listed before: "
      f"{n_files(before['identification']['source_calibration_file_list']) - 1}; listed after: {len(listed) - 1}")
assert listed[0].startswith("grid_latlon_lookup") and sorted(listed[1:]) == staged

# bounding box: outer pixel edges, the same numbers as the DISP product (same grid)
box_after = tuple(float(v) for v in ident["product_bounding_box"].strip("()").split(","))
box_disp = tuple(float(v) for v in disp["identification"]["product_bounding_box"].split(","))
box_before = tuple(float(v) for v in before["identification"]["product_bounding_box"].strip("()").split(","))
print("product_bounding_box  DISP  :", box_disp)
print("                      before:", box_before, "(pixel centres: 15 m inside on every side)")
print("                      after :", box_after)
assert box_after == box_disp

# bounding polygon: lon/lat; the raster outline contains DISP's valid-data footprint
poly_after, poly_disp = wkt.loads(ident["bounding_polygon"]), wkt.loads(disp["identification"]["bounding_polygon"])
print("bounding_polygon bounds (lon/lat)  DISP footprint:", np.round(poly_disp.bounds, 4))
print("                                   after (raster):", np.round(poly_after.bounds, 4))
print("                                   before        :", np.round(wkt.loads(before["identification"]["bounding_polygon"]).bounds, 1), "(UTM metres)")
outside = poly_disp.difference(poly_after).area / poly_disp.area
print(f"                                   DISP footprint outside the raster outline: {outside:.4%} of its area")
assert outside < 1e-3
assert -180 <= poly_after.bounds[0] <= 180 and -90 <= poly_after.bounds[1] <= 90

# nodata count: the product's own NaNs
with h5py.File(FIXED_NC) as f:
    own_nans = int(np.isnan(f["calibration"][0]).sum())
with h5py.File(GOLDEN_NC) as f:
    golden_nans = int(np.isnan(f["calibration"][0]).sum())
print(f"nodata_pixel_count  DISP displacement: {disp['identification']['nodata_pixel_count']:,}")
print(f"                    before: {before['identification']['nodata_pixel_count']:,} written, {golden_nans:,} NaNs actually in /calibration")
print(f"                    after : {ident['nodata_pixel_count']:,} written, {own_nans:,} NaNs actually in /calibration")
assert ident["nodata_pixel_count"] == own_nans

for k in ("reference_datetime", "secondary_datetime", "processing_start_datetime"):
    pd.to_datetime(ident[k], format="%Y-%m-%dT%H:%M:%SZ")  # raises if not ISO 8601 with Z
print("datetimes: all three are ISO 8601 UTC with a Z suffix")
assert not any("example.com" in str(v) for v in ident.values())
print("no example.com URL left in /identification")

station files staged: 157; listed before: 157; listed after: 157
product_bounding_box  DISP  : (71970.0, 3153930.0, 355890.0, 3385920.0)
                      before: (71970.0, 3153930.0, 355890.0, 3385920.0) (pixel centres: 15 m inside on every side)
                      after : (71970.0, 3153930.0, 355890.0, 3385920.0)
bounding_polygon bounds (lon/lat)  DISP footprint: [-97.3726  28.5028 -94.5353  30.5291]
                                   after (raster): [-97.4596  28.4418 -94.4725  30.5972]
                                   before        : [-97.5  28.4 -94.5  30.6] (UTM metres)
                                   DISP footprint outside the raster outline: 0.0000% of its area


nodata_pixel_count  DISP displacement: 25,750,140
                    before: 24,789,047 written, 24,789,047 NaNs actually in /calibration
                    after : 24,789,047 written, 24,789,047 NaNs actually in /calibration
datetimes: all three are ISO 8601 UTC with a Z suffix
no example.com URL left in /identification


## 4. The embedded PGE runconfig

Before: the Python `repr` of the internal `CalibrationWorkflow` model. After: the text of
the `RunConfig` YAML that `cal-disp run` was given — it can be loaded back as a `RunConfig`.

In [9]:
def try_reload(text):
    """Load the embedded text back as a PGE RunConfig."""
    with tempfile.NamedTemporaryFile("w", suffix=".yaml", dir=SCRATCH, delete=True) as tmp:
        tmp.write(text)
        tmp.flush()
        try:
            return RunConfig.from_yaml_file(Path(tmp.name))
        except Exception as exc:  # shown below
            return f"{type(exc).__name__}: {str(exc).splitlines()[0][:150]}"


pge_before = before["metadata"]["pge_runconfig"]
pge_after = after["metadata"]["pge_runconfig"]
print("=== before (golden), first 420 characters ===")
print(pge_before[:420])
print("\nreload as RunConfig ->", try_reload(pge_before))

=== before (golden), first 420 characters ===
# REQUIRED: A PGE (Product Generation Executive) run configuration.
cal_disp_workflow:
  # REQUIRED: Configuration for required input files..
  #   Type: Object.
  input_file_group:
    # REQUIRED: Path to DISP file.
    #   Type: string.
    disp_file: 
      test_golden/input_data/disp/OPERA_L3_DISP-S1_IW_F08882_VV_20220111T002651Z_20220722T002657Z_v1.0_20251027T005420Z.nc
    # REQUIRED: Frame ID of the DISP frame



reload as RunConfig -> ValidationError: 1 validation error for RunConfig


In [10]:
print("=== after (fixed), first 900 characters ===")
print(pge_after[:900])
reloaded = try_reload(pge_after)
assert isinstance(reloaded, RunConfig), reloaded
run_yaml = (SCRATCH / "fixed_run" / "work" / "runconfig.yaml").read_text()
assert pge_after == run_yaml
print("\nreload as RunConfig -> OK; identical to the runconfig file that was run:", pge_after == run_yaml)
print("frame_id:", reloaded.input_file_group.frame_id)
print("output_options:", reloaded.output_options.model_dump())

=== after (fixed), first 900 characters ===
# REQUIRED: A PGE (Product Generation Executive) run configuration.
cal_disp_workflow:
  # REQUIRED: Configuration for required input files..
  #   Type: Object.
  input_file_group:
    # REQUIRED: Path to DISP file.
    #   Type: string.
    disp_file: 
      /u/aurora-r0/govorcin/01_OPERA/VLM/DISP_CAL/gamma_release/cal-disp/test_golden/input_data/disp/OPERA_L3_DISP-S1_IW_F08882_VV_20220111T002651Z_20220722T002657Z_v1.0_20251027T005420Z.nc
    # REQUIRED: Frame ID of the DISP frame.
    #   Type: integer.
    frame_id: 8882
    # REQUIRED: Path to UNR grid lookup table (grid_latlon_lookup_v0.2.txt).
    #   Type: string.
    unr_grid_latlon_file: 
      /u/aurora-r0/govorcin/01_OPERA/VLM/DISP_CAL/gamma_release/cal-disp/test_golden/input_data/gnss/grid_latlon_lookup_v0.3.txt
    # REQUIRED: Directory containing UNR .tenv8 timeseries files.
    #   Type: string.
    unr_timeseries_dir: 
 

reload as RunConfig -> OK; identical to the runconfig 

## Summary

* Platform, orbit, track, pass and look direction, instrument, band, satellite names, DEM,
  imaging geometry, static-layers and source-data URLs and the CEOS ARD fields now come from
  the DISP input (table in section 2); a missing field gives a WARNING and `unknown` / `-1`.
* `processing_facility`, `product_data_access`, `static_layers_data_access` and
  `source_data_access` are `output_options` fields of the runconfig (the last two default to
  the DISP input's values). The default `product_data_access` is an ASF search URL built by
  analogy with the DISP-S1 one and should be confirmed once the product has a landing page.
* `bounding_polygon` is lon/lat, `product_bounding_box` uses the outer pixel edges (equal to
  DISP's), `nodata_pixel_count` is counted on the calibration layer, every staged station
  file is listed, datetimes are ISO 8601 with `Z`.
* `/metadata/pge_runconfig` is the RunConfig YAML that was run and loads back as a `RunConfig`.